# Notebook 1: KFC Trustpilot Review Collection

## Purpose
Scrape customer reviews from **https://www.trustpilot.com/review/kfc.co.uk** using Python `requests` and `BeautifulSoup`.

## What this notebook does
1. Sends HTTP requests to each page of the KFC Trustpilot listing
2. Parses embedded JSON (`__NEXT_DATA__`) to extract structured review data
3. Collects: Review Title, Review Text, Star Rating, Date of Experience, Author
4. Filters reviews to a 1-year date range (27 Mar 2025 - 27 Mar 2026)
5. Removes duplicates and saves to CSV

## Output
`kfc_trustpilot_reviews_2025_2026.csv`

## Prerequisites
```
pip install requests beautifulsoup4 pandas
```

## No API Key Required
Trustpilot reviews are publicly visible web pages. We use `requests` + `BeautifulSoup` to parse the HTML. No login, API key, or registration is needed.

## Step 0 - Import Libraries

In [1]:
!pip install requests beautifulsoup4 pandas
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
from datetime import datetime, timezone

print("Libraries loaded.")

Libraries loaded.


## Step 1 - Configuration

In [3]:
# Trustpilot URL for KFC UK
BASE_URL = "https://www.trustpilot.com/review/kfc.co.uk"

# Date range: 1 full year
START_DATE = datetime(2025, 3, 27, tzinfo=timezone.utc)
END_DATE   = datetime(2026, 3, 27, tzinfo=timezone.utc)

# Output filename
OUTPUT_FILE = "kfc_trustpilot_reviews_2025_2026.csv"

# HTTP headers to mimic a real browser (reduces chance of being blocked)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}

# Delay between page requests (seconds) - be respectful to the server
REQUEST_DELAY = 2

print(f"Target: {BASE_URL}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")

Target: https://www.trustpilot.com/review/kfc.co.uk
Date range: 2025-03-27 to 2026-03-27


## Step 2 - Find Total Number of Review Pages

Trustpilot paginates reviews (typically 20 per page). We fetch page 1 to discover the total number of pages.

In [4]:
def get_total_pages():
    """Fetch page 1 and extract total page count from the embedded JSON."""
    response = requests.get(f"{BASE_URL}?page=1", headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    # Trustpilot embeds data in a <script id='__NEXT_DATA__'> JSON blob
    script_tag = soup.find("script", {"id": "__NEXT_DATA__"})
    if script_tag:
        try:
            data = json.loads(script_tag.string)
            total = data["props"]["pageProps"]["filters"]["pagination"]["totalPages"]
            return total
        except (KeyError, TypeError):
            pass

    # Fallback: check pagination nav
    pagination = soup.find("nav", {"aria-label": "Pagination"})
    if pagination:
        nums = [int(a.text) for a in pagination.find_all("a") if a.text.strip().isdigit()]
        if nums:
            return max(nums)
    return 1

total_pages = get_total_pages()
print(f"Total review pages found: {total_pages}")

Total review pages found: 298


## Step 3 - Define the Review Extraction Function

Trustpilot embeds all review data in a `<script id="__NEXT_DATA__">` JSON blob. This is **more reliable** than parsing HTML class names, which Trustpilot changes frequently.

If the JSON approach fails, we fall back to parsing the HTML directly.

In [5]:
def extract_reviews_from_page(page_number):
    """
    Fetch one Trustpilot page and extract all reviews.

    Returns list of dicts with:
        Review Title, Review Text, Star Rating, Date, Author
    """
    url = f"{BASE_URL}?page={page_number}"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"  WARNING: Page {page_number} returned status {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    reviews = []

    # --- METHOD 1: Parse from __NEXT_DATA__ JSON (preferred) ---
    script_tag = soup.find("script", {"id": "__NEXT_DATA__"})
    if script_tag:
        try:
            data = json.loads(script_tag.string)
            review_list = data["props"]["pageProps"]["reviews"]

            for r in review_list:
                rating = r.get("rating", 0)
                title = r.get("title", "")
                text = r.get("text", "")

                # Prefer date of experience; fall back to published date
                date_str = r.get("dates", {}).get("experiencedDate", "")
                if not date_str:
                    date_str = r.get("dates", {}).get("publishedDate", "")

                author = r.get("consumer", {}).get("displayName", "Anonymous")

                reviews.append({
                    "Review Title": title,
                    "Review Text": text,
                    "Star Rating": rating,
                    "Date": date_str[:10] if date_str else "",
                    "Author": author,
                })
            return reviews
        except (KeyError, TypeError, json.JSONDecodeError):
            pass

    # --- METHOD 2: Fallback - parse HTML directly ---
    review_cards = soup.find_all("article", attrs={"data-service-review-card-paper": True})
    if not review_cards:
        review_cards = soup.find_all("div", class_=lambda c: c and "reviewCard" in str(c))

    for card in review_cards:
        title_tag = card.find("h2")
        title = title_tag.get_text(strip=True) if title_tag else ""

        text_tag = card.find("p", attrs={"data-service-review-text-typography": True})
        text = text_tag.get_text(strip=True) if text_tag else ""

        rating = 0
        img_tag = card.find("img", alt=lambda a: a and "Rated" in str(a))
        if img_tag:
            try:
                rating = int(img_tag["alt"].split(" ")[1])
            except (IndexError, ValueError):
                pass

        time_tag = card.find("time")
        date_str = time_tag.get("datetime", "")[:10] if time_tag else ""

        author_tag = card.find("span", attrs={"data-consumer-name-typography": True})
        author = author_tag.get_text(strip=True) if author_tag else "Anonymous"

        reviews.append({
            "Review Title": title,
            "Review Text": text,
            "Star Rating": rating,
            "Date": date_str,
            "Author": author,
        })

    return reviews

print("Review extraction function defined.")

Review extraction function defined.


## Step 4 - Scrape All Pages

Loop through every page, extract reviews, and print progress updates.

In [6]:
all_reviews = []

print(f"Scraping {total_pages} pages...")
for page in range(1, total_pages + 1):
    reviews = extract_reviews_from_page(page)
    all_reviews.extend(reviews)

    if page % 10 == 0 or page == total_pages:
        print(f"  Page {page}/{total_pages} - {len(all_reviews)} reviews so far")

    time.sleep(REQUEST_DELAY)

print(f"\nTotal reviews scraped (before date filter): {len(all_reviews)}")

Scraping 298 pages...
  Page 10/298 - 200 reviews so far
  Page 20/298 - 200 reviews so far
  Page 30/298 - 200 reviews so far
  Page 40/298 - 200 reviews so far
  Page 50/298 - 200 reviews so far
  Page 60/298 - 200 reviews so far
  Page 70/298 - 200 reviews so far
  Page 80/298 - 200 reviews so far
  Page 90/298 - 200 reviews so far
  Page 100/298 - 200 reviews so far
  Page 110/298 - 200 reviews so far
  Page 120/298 - 200 reviews so far
  Page 130/298 - 200 reviews so far
  Page 140/298 - 200 reviews so far
  Page 150/298 - 200 reviews so far
  Page 160/298 - 200 reviews so far
  Page 170/298 - 200 reviews so far
  Page 180/298 - 200 reviews so far
  Page 190/298 - 200 reviews so far
  Page 200/298 - 200 reviews so far
  Page 210/298 - 200 reviews so far
  Page 220/298 - 200 reviews so far
  Page 230/298 - 200 reviews so far
  Page 240/298 - 200 reviews so far
  Page 250/298 - 200 reviews so far
  Page 260/298 - 200 reviews so far
  Page 270/298 - 200 reviews so far
  Page 280/298 

## Step 5 - Filter by Date Range, Deduplicate, and Save

In [8]:
df = pd.DataFrame(all_reviews)

if df.empty:
    print("ERROR: No reviews collected.")
    print("Trustpilot may have blocked the request. Try increasing REQUEST_DELAY.")
else:
    # Convert Date to datetime for filtering and make it timezone-aware
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)

    # Filter to date range
    before = len(df)
    df = df[(df["Date"] >= START_DATE) & (df["Date"] <= END_DATE)]
    print(f"Date filter: {before} -> {len(df)} reviews")

    # Convert Date back to string
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    # Deduplicate
    before = len(df)
    df.drop_duplicates(subset=["Author", "Review Text"], inplace=True)
    print(f"Deduplication: {before} -> {len(df)} reviews")

    # Save
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\nSaved to: {OUTPUT_FILE}")

    print(f"\n--- Dataset Summary ---")
    print(f"Total reviews:   {len(df)}")
    print(f"Date range:      {df['Date'].min()} to {df['Date'].max()}")
    print(f"Avg star rating: {df['Star Rating'].mean():.2f}")
    print(f"\nRating distribution:")
    print(df["Star Rating"].value_counts().sort_index())
    df.head()

Date filter: 200 -> 200 reviews
Deduplication: 200 -> 200 reviews

Saved to: kfc_trustpilot_reviews_2025_2026.csv

--- Dataset Summary ---
Total reviews:   200
Date range:      2025-11-11 to 2026-03-27
Avg star rating: 2.23

Rating distribution:
Star Rating
1    125
2     13
3      4
4      6
5     52
Name: count, dtype: int64
